# Error Severity Triage

**Last updated:** 2026-04-15 12:00 PT

**Track:** Learning | **Sub-Ability:** Concept formation (category boundaries)

Can the model learn a new tool's severity classification system from a handful of labeled log lines and apply it to a novel line?

The twist: models have a strong prior that anything labeled "error" is blocking. This benchmark tests whether a model can override that prior and correctly distinguish `RETRYABLE_ERROR` from `TERMINAL_ERROR` using only the evidence in the labeled examples.

**Classes:** `INFO`, `WARNING`, `RETRYABLE_ERROR`, `TERMINAL_ERROR`

**Difficulty grid:** complexity (single_tag / tag_plus_code / contextual) × evidence (few=5 / mid=8 / many=12) × 3 seeds = 27 items

In [ ]:
import kaggle_benchmarks as kbench
import pandas as pd
import random
import re
import json
import time

print('Available models:', list(kbench.llms.keys()))
print('Default model:', kbench.llm)

In [ ]:
VALID_LABELS = ['INFO', 'WARNING', 'RETRYABLE_ERROR', 'TERMINAL_ERROR']

def strip_thinking(text: str) -> str:
    """Remove <think>...</think> blocks from reasoning model output."""
    if '</think>' in text:
        return text.split('</think>', 1)[1].strip()
    return text.strip()

def extract_severity_label(response: str) -> str:
    """Extract one of INFO / WARNING / RETRYABLE_ERROR / TERMINAL_ERROR from model output.

    Strategy: clean markdown formatting (but preserve underscores since labels contain them),
    then scan lines (bottom-up) for an exact token match. Checks compound labels
    (RETRYABLE_ERROR / TERMINAL_ERROR) first so they're not shadowed by partial matches.
    Falls back to the first match anywhere in the response. Returns '' if nothing matches.
    """
    text = strip_thinking(response)
    # Strip backticks and markdown emphasis but KEEP underscores (labels contain them)
    text = re.sub(r'[`*]', '', text)
    # Check order: longest/most-specific labels first
    check_order = ['RETRYABLE_ERROR', 'TERMINAL_ERROR', 'WARNING', 'INFO']
    lines = [l.strip() for l in text.strip().split('\n') if l.strip()]
    # Scan from the end (models often put the final answer last)
    for line in reversed(lines):
        up = line.upper()
        for lab in check_order:
            # Treat '_' as a word char (it is), so we use explicit boundary chars.
            # Match when label is at start/end of string or surrounded by non-alnum/non-underscore.
            if re.search(rf'(?:^|[^A-Z0-9_]){lab}(?:$|[^A-Z0-9_])', up):
                return lab
    # Fallback: first match anywhere in the full text
    up_all = text.upper()
    for lab in check_order:
        if re.search(rf'(?:^|[^A-Z0-9_]){lab}(?:$|[^A-Z0-9_])', up_all):
            return lab
    return ''

In [ ]:
# === Load dataset from Kaggle ===
import os, glob
candidates = glob.glob('/kaggle/input/**/error_severity_triage_dataset.csv', recursive=True)
if candidates:
    csv_path = candidates[0]
else:
    csv_path = '/kaggle/input/error_severity_triage_dataset.csv'
print(f'Loading from: {csv_path}')
dataset = pd.read_csv(csv_path)
print(f'Dataset: {len(dataset)} items')
print('Class distribution:')
print(dataset['expected'].value_counts())

In [ ]:
PRE_P = """You are triaging log output from an unfamiliar CLI tool. The tool uses its own conventions to mark severity of each line.

Here are labeled example lines from this tool (line    -> SEVERITY):

{material}

Classify this new line from the same tool:

  {test_input}

Reply with exactly ONE of these labels (no other text):
INFO
WARNING
RETRYABLE_ERROR
TERMINAL_ERROR"""

STUDY_P = """You are studying log output from an unfamiliar CLI tool to learn its severity conventions.

Labeled example lines (line    -> SEVERITY):

{material}

The possible severity classes are: INFO, WARNING, RETRYABLE_ERROR, TERMINAL_ERROR.

Create a systematic study guide for this tool's conventions:
1. For EACH class (INFO, WARNING, RETRYABLE_ERROR, TERMINAL_ERROR), list which surface features (prefix tags, error codes, keywords, message patterns) distinguish it.
2. Pay special attention to what separates RETRYABLE_ERROR from TERMINAL_ERROR. Do NOT assume 'error' means terminal — use only the evidence from the labeled examples. List concrete cues (code prefixes, numeric ranges, body keywords) that distinguish them in THIS tool.
3. Write a short decision procedure: given a new line, which features do you check in what order to assign a class?
4. Verify your rules by re-classifying every labeled example and confirming each one matches its given label. If any don't match, REVISE your rules.

Show all verification work."""

POST_P = """You previously studied an unfamiliar CLI tool's severity conventions and wrote these notes:

--- YOUR NOTES ---
{notes}
--- END NOTES ---

Original labeled examples (line    -> SEVERITY):

{material}

Classify this new line from the same tool:

  {test_input}

Reply with exactly ONE of these labels (no other text):
INFO
WARNING
RETRYABLE_ERROR
TERMINAL_ERROR"""

## Evaluation

In [ ]:
all_results = []

@kbench.task(name='error_severity_triage',
             description='Pre/post learning severity classification of log lines from an invented CLI tool')
def error_severity_triage(llm) -> float:
    model_name = str(llm)
    correct = 0
    total = len(dataset)

    for _, row in dataset.iterrows():
        material = row['material']
        test_input = row['test_input']
        expected = row['expected']
        complexity = row['complexity']
        evidence = row['evidence']
        difficulty_label = row['difficulty_label']
        seed = int(row['seed'])
        item_desc = row['item_desc']
        ts = time.strftime('%Y-%m-%d %H:%M:%S')
        tid = f'{complexity}_{evidence}_{seed}'

        # --- PRE: baseline without study ---
        t0 = time.time()
        pre_raw = llm.prompt(PRE_P.format(material=material, test_input=test_input))
        pre_time = time.time() - t0
        pre_extracted = extract_severity_label(pre_raw)
        pre_correct = pre_extracted == expected

        # --- STUDY: create classification guide ---
        t0 = time.time()
        study_raw = llm.prompt(STUDY_P.format(material=material))
        study_time = time.time() - t0
        notes = strip_thinking(study_raw)

        # --- POST: classify with study notes ---
        t0 = time.time()
        post_raw = llm.prompt(POST_P.format(notes=notes, material=material, test_input=test_input))
        post_time = time.time() - t0
        post_extracted = extract_severity_label(post_raw)
        post_correct = post_extracted == expected

        if post_correct:
            correct += 1

        all_results.append({
            'timestamp': ts, 'model': model_name, 'task_id': tid,
            'complexity': complexity, 'evidence': evidence, 'difficulty_label': difficulty_label,
            'seed': int(seed), 'item_desc': item_desc,
            'test_input': test_input, 'expected': expected,
            'pre_raw': pre_raw, 'pre_extracted': pre_extracted, 'pre_correct': int(pre_correct),
            'study_notes': notes,
            'post_raw': post_raw, 'post_extracted': post_extracted, 'post_correct': int(post_correct),
            'pre_time_s': pre_time, 'study_time_s': study_time, 'post_time_s': post_time
        })

        b, l = 'Y' if pre_correct else 'N', 'Y' if post_correct else 'N'
        print(f'[{model_name:40s}] [{difficulty_label:22s}] exp={expected:16s}  '
              f'pre={pre_extracted:16s}({b})  post={post_extracted:16s}({l})')
        kbench.assertions.assert_equal(post_extracted, expected)

    score = correct / total
    print(f'\nScore: {correct}/{total} = {score:.4f}')
    return score

try:
    runs = error_severity_triage.evaluate(llm=[kbench.llm],
                                          n_jobs=1, timeout=3600, max_attempts=1)
    print(f'\nDone: {len(runs.as_dataframe())} items')
except Exception as e:
    print(f'SDK post-processing error (non-fatal): {e}')

## Results

In [ ]:
results = pd.DataFrame(all_results)
print(f'Total runs: {len(results)}\n')

pre_acc = results['pre_correct'].mean()
post_acc = results['post_correct'].mean()
gain = post_acc - pre_acc

print('=' * 60)
print(f'Model: {results["model"].iloc[0] if len(results) > 0 else "N/A"}')
print(f'Pre-learning accuracy:  {pre_acc:.1%}')
print(f'Post-learning accuracy: {post_acc:.1%}')
print(f'Learning gain:          {gain:+.1%}')
print()

print('By Difficulty:')
print('-' * 60)
for label in sorted(results['difficulty_label'].unique()):
    sub = results[results['difficulty_label'] == label]
    b = sub['pre_correct'].mean()
    l = sub['post_correct'].mean()
    g = l - b
    print(f'  {label:25s}  pre={b:.1%}  post={l:.1%}  gain={g:+.1%}  (n={len(sub)})')

print()
print('By Expected Class (post-learning):')
print('-' * 60)
for cls in ['INFO', 'WARNING', 'RETRYABLE_ERROR', 'TERMINAL_ERROR']:
    sub = results[results['expected'] == cls]
    if len(sub) == 0:
        continue
    acc = sub['post_correct'].mean()
    print(f'  {cls:18s}  post={acc:.1%}  (n={len(sub)})')

print()
print(f'Timing: pre={results["pre_time_s"].mean():.1f}s  study={results["study_time_s"].mean():.1f}s  post={results["post_time_s"].mean():.1f}s')

In [ ]:
print('STUDY NOTES INSPECTION')
print('=' * 60)
for _, row in results.iterrows():
    b = 'Y' if row['pre_correct'] else 'N'
    l = 'Y' if row['post_correct'] else 'N'
    print(f'\n--- [{row["difficulty_label"]}] seed={row["seed"]} ---')
    print(f'Item: {row["item_desc"]}')
    print(f'Test line: {row["test_input"]}')
    print(f'Expected: {row["expected"]}')
    print(f'Pre: {row["pre_extracted"]} ({b})  Post: {row["post_extracted"]} ({l})')
    print(f'Notes (first 300 chars): {str(row["study_notes"])[:300]}')
    print('...' if len(str(row['study_notes'])) > 300 else '')

In [ ]:
!pip install -q matplotlib
import matplotlib.pyplot as plt

labels = sorted(results['difficulty_label'].unique())
pre_scores = [results[results['difficulty_label']==l]['pre_correct'].mean() for l in labels]
post_scores = [results[results['difficulty_label']==l]['post_correct'].mean() for l in labels]

x = range(len(labels))
width = 0.35
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

ax1 = axes[0]
ax1.bar([i-width/2 for i in x], pre_scores, width, label='Pre-learning', color='#4C72B0')
ax1.bar([i+width/2 for i in x], post_scores, width, label='Post-learning', color='#55A868')
ax1.set_ylabel('Accuracy')
ax1.set_title('Error Severity Triage: Pre vs Post-Learning')
ax1.set_xticks(list(x))
ax1.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
ax1.legend()
ax1.set_ylim(0, 1.05)

ax2 = axes[1]
gains = [p-b for b,p in zip(pre_scores, post_scores)]
colors = ['#55A868' if g>=0 else '#C44E52' for g in gains]
ax2.bar(range(len(labels)), gains, color=colors)
ax2.set_ylabel('Learning Gain')
ax2.set_title('Learning Gain by Difficulty')
ax2.set_xticks(list(x))
ax2.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
ax2.axhline(y=0, color='gray', linestyle='--', linewidth=0.8)

plt.tight_layout()
plt.savefig('error_severity_triage_results.png', dpi=150, bbox_inches='tight')
plt.show()